In [3]:
import os
import json
import numpy as np

import jellyfish
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict


SEQUENCE_LENGTH = 512
SEQUENCES_PER_SAMPLE = 10


BASE = "/Volumes/New Volume/malware-detection-dataset/opcodes"

In [4]:
labels = json.load(open(os.path.join(BASE, "labels.json"), "r"))
extracted_opcodes = list(labels.keys())

In [ ]:
def get_instructions(paths):
    instructions = set()
    loop = tqdm(paths, desc="Creating Vocab")
    for path in loop:
        obj = json.load(open(path, "r"))
        [instructions.add(instr) for instr in obj]
        loop.set_postfix_str(f"{len(instructions)} Unique Opcodes")

    return instructions


unique_instructions = (
    get_instructions(extracted_opcodes)
    if not os.path.exists("./vocab.json")
    else json.load(open("./vocab.json", "r"))
)

In [ ]:
def extract_opcode_sequences(
    path: os.PathLike,
    sequence_length: int = SEQUENCE_LENGTH,
    n_sequences: int = SEQUENCES_PER_SAMPLE,
    tolerance: int = 10,
):
    full_sequence = json.load(open(path, "r"))

    if len(full_sequence) == 0:
        raise Exception("No data to extract")

    sequences = []
    attempts = 0

    while len(sequences) < n_sequences:
        attempts += 1

        if attempts >= 1000:
            raise Exception("Max number of attempts reached")

        start_idx = np.random.randint(0, len(full_sequence))
        sequence = full_sequence[start_idx : start_idx + sequence_length]

        self_sim = np.array(
            [[jellyfish.levenshtein_distance(a, b) for b in sequence] for a in sequence]
        ).mean()

        if np.isclose(self_sim, 0, 0, tolerance):
            continue

        while len(sequence) < sequence_length:
            sequence.append("[PAD]")

        sequences.append(sequence)

    return sequences


def process_file(path: os.PathLike) -> Dict[str, List[List[str]]]:
    """Wrapper function for extracting sequences with error handling."""
    try:
        return {path: extract_opcode_sequences(path)}
    except Exception as e:
        return {path: f"Failed: {str(e)}"}


def extract_all_sequences(
    file_paths: List[os.PathLike], max_workers: int = 4
) -> Dict[str, List[List[str]]]:
    """
    Extract opcode sequences from multiple files using multithreading.

    Args:
        file_paths: List of file paths to process.
        max_workers: Number of threads to use.

    Returns:
        A dictionary mapping file paths to extracted sequences.
    """
    sequence_dict = {}
    failed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_path = {
            executor.submit(process_file, path): path for path in file_paths
        }

        for future in tqdm(
            as_completed(future_to_path),
            total=len(file_paths),
            desc="Generating Opcode Sequences",
        ):
            path = future_to_path[future]
            result = future.result()
            if "Failed" in result[path]:
                failed += 1
            sequence_dict.update(result)

    print(f"Processing completed with {failed} failures.")
    return sequence_dict


sequence_dict = extract_all_sequences(extracted_opcodes, max_workers=8)

Generating Opcode Sequences:   0%|          | 37/9880 [06:06<27:05:45,  9.91s/it]


In [16]:
processed_base = os.path.join(BASE, "processed_data")
os.makedirs(processed_base)

for path, sequences in tqdm(sequence_dict.items()):
    filename = path.split("/")[-1]
    full_path = os.path.join(processed_base, filename)
    sequences = [[instr.split(" ")[0] for instr in sequence] for sequence in sequences]

    with open(full_path, "w") as fp:
        json.dump(sequences, fp)

100%|██████████| 9786/9786 [00:10<00:00, 901.66it/s]
